# 01 — Exploratory Data Analysis

This notebook loads the cleaned Manhattan → JFK/LGA taxi dataset and explores duration, late-rate, and pickup-zone patterns.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/processed/taxi_clean_for_modeling.csv')
df = pd.read_csv(DATA, parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])

if 'duration_min' not in df.columns:
    df['duration_min'] = df['trip_duration_min']

df.shape, df.head()

## Data Quality Checks

A good transportation model starts with realistic data checks: impossible trip durations, zero distances, airport destination consistency, and pickup/dropoff timestamp validity.


In [ ]:
summary = {
    'rows': len(df),
    'min_duration': df['duration_min'].min(),
    'median_duration': df['duration_min'].median(),
    'max_duration': df['duration_min'].max(),
    'late_rate': df['is_late'].mean() if 'is_late' in df.columns else None,
}
summary

## Duration Patterns by Hour and Airport

Airport travel is not just a distance problem. The same pickup zone behaves differently by time of day, weekday, and airport destination.


In [ ]:
hour_airport = df.groupby(['pickup_hour','airport'])['duration_min'].median().unstack()
ax = hour_airport.plot(marker='o', figsize=(9,4.5))
ax.set_title('Median Airport Taxi Duration by Pickup Hour')
ax.set_xlabel('Pickup hour')
ax.set_ylabel('Median duration (min)')
plt.show()

In [ ]:
late_hour = df.groupby(['pickup_hour','airport'])['is_late'].mean().unstack()
ax = (late_hour * 100).plot(marker='o', figsize=(9,4.5))
ax.set_title('Late-Trip Rate by Pickup Hour')
ax.set_xlabel('Pickup hour')
ax.set_ylabel('Late rate (%)')
plt.show()

## Pickup Zone Concentration

High-volume zones matter more for product design because a timing advisor should work well where travelers actually request airport rides.


In [ ]:
zone_col = 'PU_Zone' if 'PU_Zone' in df.columns else 'pickup_zone'
top_zones = df[zone_col].value_counts().head(15)
ax = top_zones.sort_values().plot(kind='barh', figsize=(9,6))
ax.set_title('Top Manhattan Pickup Zones for Airport Trips')
ax.set_xlabel('Trips')
plt.show()